# Deploy Vast.ai — destroy after failed upload, but try sending logs first

In [ ]:
%pip -q install paramiko requests

In [ ]:
from pathlib import Path
import base64, json, os, re, shlex
import paramiko

CONFIG = {
    "ssh_host": "ssh9.vast.ai",
    "ssh_port": 16453,
    "ssh_user": "root",
    "ssh_key_path": os.path.expanduser("~/.ssh/id_ed25519"),
    "ssh_key_passphrase": None,

    "vast_instance_id": "PASTE_VAST_INSTANCE_ID_HERE",
    "auto_stop_after_success": True,
    "destroy_instead_of_stop": True,
    "destroy_on_upload_failure": True,
    "upload_retry_attempts": 3,

    "fork_repo_url": "https://github.com/YOUR_GITHUB_USER/chatterbox.git",
    "fork_commit_sha": "PASTE_FULL_COMMIT_SHA_HERE",

    "hf_model_repo_id": "ResembleAI/chatterbox",
    "hf_model_revision": "PASTE_HF_REVISION_OR_COMMIT_HERE",

    "remote_base_dir": "/workspace/chbx_ja_batch",
    "server_port": 7860,
    "hf_cache_dir": "/workspace/.cache/huggingface",

    "rclone_remote": "gdrive:tts-runs-chatterbox",
    "rclone_config_b64": "",

    "cloudflared_bin": "/usr/local/bin/cloudflared",
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
class SSHRemote:
    def __init__(self, host, port, user, key_path, passphrase=None, timeout=30):
        self.host = host; self.port = int(port); self.user = user; self.key_path = key_path; self.passphrase = passphrase; self.timeout = timeout; self.client = None; self.sftp = None
    def connect(self):
        key = None; errors = []
        for cls in (paramiko.Ed25519Key, paramiko.RSAKey, paramiko.ECDSAKey):
            try:
                key = cls.from_private_key_file(self.key_path, password=self.passphrase); break
            except Exception as e:
                errors.append(str(e))
        if key is None: raise RuntimeError("Failed to load SSH key:\n" + "\n".join(errors))
        cli = paramiko.SSHClient(); cli.set_missing_host_key_policy(paramiko.AutoAddPolicy()); cli.connect(hostname=self.host, port=self.port, username=self.user, pkey=key, timeout=self.timeout); self.client = cli; self.sftp = cli.open_sftp(); return self
    def run(self, cmd, env=None, check=True, get_pty=False, timeout=None):
        if env:
            exports = " ".join(f"{k}={shlex.quote(str(v))}" for k,v in env.items()); cmd = f"export {exports}; {cmd}"
        stdin, stdout, stderr = self.client.exec_command(cmd, get_pty=get_pty, timeout=timeout)
        out = stdout.read().decode("utf-8", errors="replace"); err = stderr.read().decode("utf-8", errors="replace"); code = stdout.channel.recv_exit_status()
        if check and code != 0: raise RuntimeError(f"Remote command failed ({code})\nCMD: {cmd}\nSTDOUT:\n{out}\nSTDERR:\n{err}")
        return {"code": code, "stdout": out, "stderr": err}
    def mkdir_p(self, remote_path):
        import posixpath
        parts = remote_path.strip("/").split("/"); cur = ""
        for part in parts:
            cur = posixpath.join(cur, part); rp = "/" + cur.lstrip("/")
            try: self.sftp.stat(rp)
            except FileNotFoundError: self.sftp.mkdir(rp)
    def write_text(self, remote_path, text):
        import posixpath
        self.mkdir_p(posixpath.dirname(remote_path))
        with self.sftp.open(remote_path, "w") as f: f.write(text)
    def write_bytes(self, remote_path, data):
        import posixpath
        self.mkdir_p(posixpath.dirname(remote_path))
        with self.sftp.open(remote_path, "wb") as f: f.write(data)

remote = SSHRemote(CONFIG["ssh_host"], CONFIG["ssh_port"], CONFIG["ssh_user"], CONFIG["ssh_key_path"], CONFIG["ssh_key_passphrase"]).connect()
print("SSH connected")

In [ ]:
bundle_files = {
    "README.md": '# Chatterbox — destroy after failed upload, but try sending logs first\n\nThis version adds:\n- upload retries\n- destroy instance after failed upload retries\n- final log-only upload attempt before destroy\n',
    "requirements.lock.txt": 'gradio==5.29.0\nsoundfile==0.13.1\npandas==2.2.3\nhuggingface_hub==0.30.2\nrequests==2.32.3\nparamiko==3.5.1\nnumpy==2.2.4\nscipy==1.15.2\nlibrosa==0.11.0\npydub==0.25.1\ntorch==2.6.0\ntorchaudio==2.6.0\n',
    "RUN_METADATA.template.json": '{\n  "run_id": "",\n  "repo_url": "",\n  "repo_commit_sha": "",\n  "hf_model_repo_id": "",\n  "hf_model_revision": "",\n  "requirements_lock_sha256": "",\n  "gpu_name": "",\n  "device": "",\n  "started_utc": "",\n  "ended_utc": "",\n  "rclone_remote": "",\n  "count_items": 0,\n  "profile": {},\n  "generated_files": [],\n  "jobs": []\n}',
    "app.py": 'import datetime as dt, hashlib, json, os, platform, re, shutil, subprocess, sys, time, traceback, uuid\nfrom pathlib import Path\nimport gradio as gr, numpy as np, soundfile as sf, torch\n\nAPP_ROOT = Path(os.environ.get("APP_ROOT", Path(__file__).resolve().parent))\nDATA_ROOT = APP_ROOT / "data"\nINPUT_ROOT, OUTPUT_ROOT, LOG_ROOT, STATE_ROOT = DATA_ROOT/"inputs", DATA_ROOT/"outputs", DATA_ROOT/"logs", DATA_ROOT/"state"\nRUNTIME_LOG_DIR = APP_ROOT / "runtime_logs"\nfor p in [INPUT_ROOT, OUTPUT_ROOT, LOG_ROOT, STATE_ROOT, RUNTIME_LOG_DIR]:\n    p.mkdir(parents=True, exist_ok=True)\n\nSUPPORTED_LANGUAGES = ["ar","da","de","el","en","es","fi","fr","he","hi","it","ja","ko","ms","nl","no","pl","pt","ru","sv","sw","tr","zh"]\nDEFAULT_PROFILE = {"language":"ja","voice":"ja_female_neutral","speed":0.92,"pitch":-1.0,"temperature":0.55,"top_p":0.9,"pause_scale":1.2,"sentence_silence_ms":320,"comma_silence_ms":140,"emotion_strength":0.35,"clarity":0.95,"breathiness":0.25,"chunk_soft_limit":220,"chunk_hard_limit":280,"retry_per_chunk":2,"target_sr":24000}\nGPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nMODEL = None\n\ndef now_utc(): return dt.datetime.utcnow().isoformat() + "Z"\ndef log_line(msg:str):\n    with open(RUNTIME_LOG_DIR/"debug_app.log","a",encoding="utf-8") as f: f.write(f"[{now_utc()}] {msg}\\n")\ndef write_json(path:Path,obj):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")\ndef sha256_file(path:Path):\n    h=hashlib.sha256()\n    with open(path,"rb") as f:\n        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)\n    return h.hexdigest()\ndef shell_capture(cmd):\n    p=subprocess.run(cmd,capture_output=True,text=True)\n    return {"cmd":cmd,"rc":p.returncode,"stdout":p.stdout[-12000:],"stderr":p.stderr[-12000:]}\ndef write_environment_snapshot():\n    snap={"timestamp_utc":now_utc(),"python":sys.version,"platform":platform.platform(),"device":DEVICE,"gpu_name":GPU_NAME,"nvidia_smi":shell_capture(["bash","-lc","nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"]),"rclone_version":shell_capture(["bash","-lc","rclone version || true"]),"git_head":shell_capture(["bash","-lc","cd \\"$APP_ROOT/src\\" 2>/dev/null && git rev-parse HEAD || true"]),"env_subset":{k:os.environ.get(k,"") for k in ["APP_ROOT","SERVER_PORT","RCLONE_REMOTE","FORK_REPO_URL","FORK_COMMIT_SHA","HF_MODEL_REPO_ID","HF_MODEL_REVISION","VAST_INSTANCE_ID","DESTROY_INSTEAD_OF_STOP","DESTROY_ON_UPLOAD_FAILURE","UPLOAD_RETRY_ATTEMPTS"]}}\n    write_json(RUNTIME_LOG_DIR/"environment_snapshot.json", snap)\ndef schedule_vast_action(action:str, reason:str):\n    instance_id=os.environ.get("VAST_INSTANCE_ID","").strip() or os.environ.get("CONTAINER_ID","").strip()\n    if not instance_id:\n        log_line(f"Skip {action}; no instance id. reason={reason}"); return\n    script=APP_ROOT/f"{action}_after_run.sh"\n    script.write_text(f"#!/usr/bin/env bash\\nset -e\\nsleep 10\\npython3 -m pip -q install vastai >/dev/null 2>&1 || true\\nvastai {action} instance {instance_id} || true\\n",encoding="utf-8")\n    script.chmod(0o755); log_line(f"Scheduling {action} for instance {instance_id}; reason={reason}")\n    subprocess.Popen(["bash", str(script)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\ndef load_model():\n    global MODEL\n    if MODEL is not None: return MODEL\n    log_line("Loading Chatterbox multilingual model")\n    from chatterbox.mtl_tts import ChatterboxMultilingualTTS\n    MODEL = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)\n    write_json(RUNTIME_LOG_DIR/"model_load.json",{"timestamp_utc":now_utc(),"import_path":"chatterbox.mtl_tts.ChatterboxMultilingualTTS","factory":"ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)","device":DEVICE,"gpu_name":GPU_NAME})\n    return MODEL\ndef sanitize_name(x:str)->str:\n    x=re.sub(r"[^a-zA-Z0-9._-]+","_",(x or "").strip()); return x.strip("_") or f"item_{uuid.uuid4().hex[:8]}"\ndef split_sections(text:str):\n    parts=re.split(r"\\[(HOOK|CONTEXT|DEV|CLIMAX|END)\\]",text)\n    if len(parts)<3: return [("DEV",text.strip())]\n    out=[]\n    for i in range(1,len(parts),2):\n        body=parts[i+1].strip() if i+1 < len(parts) else ""\n        if body: out.append((parts[i],body))\n    return out or [("DEV",text.strip())]\ndef strip_markers_for_model(text:str)->str:\n    return re.sub(r"\\[(HOOK|CONTEXT|DEV|CLIMAX|END)\\]"," ",text).replace("  "," ").strip()\ndef section_profile(base,tag):\n    p=dict(base)\n    if tag=="HOOK":\n        p["speed"]=max(0.84,p["speed"]-0.04); p["emotion_strength"]=min(0.55,p["emotion_strength"]+0.05)\n    elif tag=="CONTEXT":\n        p["speed"]=min(1.00,p["speed"]+0.02); p["clarity"]=min(1.0,p["clarity"]+0.02)\n    elif tag=="CLIMAX":\n        p["speed"]=max(0.86,p["speed"]-0.03); p["emotion_strength"]=min(0.60,p["emotion_strength"]+0.07)\n    elif tag=="END":\n        p["speed"]=max(0.82,p["speed"]-0.05)\n    return p\ndef sentence_split_generic(text:str):\n    chunks=re.split(r"(?<=[。！？….!?])",text); chunks=[x.strip() for x in chunks if x.strip()]; return chunks or [text]\ndef chunk_text(text,soft_limit=220,hard_limit=280):\n    text=text.strip()\n    if len(text)<=hard_limit: return [text]\n    out=[]; buf=""\n    for sent in sentence_split_generic(text):\n        candidate=(buf+" "+sent).strip() if buf else sent\n        if len(candidate)<=soft_limit: buf=candidate; continue\n        if buf: out.append(buf); buf=sent\n        else:\n            while len(sent)>hard_limit: out.append(sent[:hard_limit]); sent=sent[hard_limit:]\n            buf=sent\n    if buf: out.append(buf)\n    return out\ndef estimate_seconds(text): return round((max(1,len(text))/11.0) * (0.55 if DEVICE=="cuda" else 3.0), 1)\ndef concat_with_silence(wavs,sr,sentence_silence_ms=320):\n    if not wavs: return np.zeros((1,),dtype=np.float32), sr\n    silence=np.zeros((int(sr*sentence_silence_ms/1000.0),),dtype=np.float32); out=[]\n    for i,w in enumerate(wavs):\n        out.append(np.asarray(w).astype(np.float32).reshape(-1))\n        if i < len(wavs)-1: out.append(silence)\n    return np.concatenate(out), sr\ndef run_rclone_copy(src_dir:Path, remote_path:str):\n    p=subprocess.run(["rclone","copy",str(src_dir),remote_path,"--create-empty-src-dirs","-P"],capture_output=True,text=True)\n    return p.returncode,p.stdout[-12000:],p.stderr[-12000:]\ndef retry_rclone_copy(src_dir:Path, remote_path:str, label:str, attempts:int, log_obj:dict):\n    hist=[]\n    for attempt in range(1,attempts+1):\n        rc,out,err=run_rclone_copy(src_dir,remote_path)\n        hist.append({"attempt":attempt,"rc":rc,"stdout":out,"stderr":err})\n        if rc==0:\n            log_obj[label]={"ok":True,"attempts":hist}; return True\n        time.sleep(min(10,attempt*2))\n    log_obj[label]={"ok":False,"attempts":hist}; return False\ndef build_jobs_table(jobs):\n    return [[i,j["name"],j["language"],len(strip_markers_for_model(j["script"])),bool(j.get("clone_wav")),estimate_seconds(strip_markers_for_model(j["script"]))] for i,j in enumerate(jobs,1)]\ndef add_job(name, language, script, clone_wav, jobs_state):\n    jobs=list(jobs_state or [])\n    if not (name or "").strip(): raise gr.Error("Name is required.")\n    if not (script or "").strip(): raise gr.Error("Script is required.")\n    if language not in SUPPORTED_LANGUAGES: raise gr.Error("Unsupported language.")\n    safe_name=sanitize_name(name); clone_path=clone_wav if clone_wav else None\n    jobs.append({"name":safe_name,"language":language,"script":script.strip(),"clone_wav":clone_path})\n    log_line(f"Added job {safe_name} lang={language} clone={bool(clone_path)}")\n    return jobs, build_jobs_table(jobs), f"{len(jobs)} job(s) in queue.", "", None\ndef remove_last_job(jobs_state):\n    jobs=list(jobs_state or [])\n    if jobs: removed=jobs.pop(); log_line(f"Removed job {removed.get(\'name\')}")\n    return jobs, build_jobs_table(jobs), f"{len(jobs)} job(s) in queue."\ndef clear_jobs():\n    log_line("Cleared queue"); return [], [], "Queue cleared."\ndef synthesize_one(item_name,text,voice_wav,profile,run_dir:Path,per_job_log_dir:Path):\n    model=load_model(); sections=split_sections(text); all_audio=[]; logs=[]\n    for tag, section_text in sections:\n        p=section_profile(profile,tag); chunks=chunk_text(section_text,int(p["chunk_soft_limit"]),int(p["chunk_hard_limit"]))\n        for chunk_idx,chunk in enumerate(chunks,1):\n            last_err=None\n            for attempt in range(int(p["retry_per_chunk"])+1):\n                try:\n                    t0=time.time(); wav=model.generate(chunk,language_id=p["language"],audio_prompt_path=voice_wav if voice_wav else None); gen_s=round(time.time()-t0,3)\n                    if torch.is_tensor(wav): wav=wav.detach().float().cpu().numpy()\n                    all_audio.append(np.asarray(wav).reshape(-1)); logs.append({"section":tag,"chunk_index":chunk_idx,"attempt":attempt+1,"status":"ok","language":p["language"],"chars":len(chunk),"generation_seconds":gen_s}); break\n                except Exception as e:\n                    last_err=traceback.format_exc(); logs.append({"section":tag,"chunk_index":chunk_idx,"attempt":attempt+1,"status":"retry" if attempt < int(p["retry_per_chunk"]) else "failed","error":str(e),"traceback":last_err[-6000:],"language":p["language"],"chars":len(chunk)}); time.sleep(1.0+attempt)\n            else:\n                write_json(per_job_log_dir/f"{item_name}_crash.json",{"error":last_err}); raise RuntimeError(f"Chunk failed permanently: {tag} #{chunk_idx}\\n{last_err}")\n    sr=getattr(model,"sr",int(profile["target_sr"])); audio,sr=concat_with_silence(all_audio,sr=sr,sentence_silence_ms=int(profile["sentence_silence_ms"])); out_wav=run_dir/f"{item_name}.wav"; sf.write(out_wav,audio,sr); return out_wav,logs\ndef generate_all(jobs_state, default_language, voice_name, profile_json, auto_upload, destroy_after_run, progress=gr.Progress(track_tqdm=False)):\n    write_environment_snapshot(); jobs=list(jobs_state or [])\n    if not jobs: raise gr.Error("Add at least one job first.")\n    started=time.time(); started_utc=now_utc(); run_id=dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8]\n    run_dir,inp_dir,log_dir,per_job_log_dir=OUTPUT_ROOT/run_id,INPUT_ROOT/run_id,LOG_ROOT/run_id,(LOG_ROOT/run_id/"jobs")\n    for p in [run_dir,inp_dir,log_dir,per_job_log_dir]: p.mkdir(parents=True, exist_ok=True)\n    profile=dict(DEFAULT_PROFILE); profile["language"]=default_language; profile["voice"]=voice_name\n    incoming=json.loads(profile_json.strip()) if profile_json.strip() else {}; profile.update(incoming)\n    write_json(log_dir/"profile_effective.json",profile); write_json(log_dir/"queued_jobs_initial.json",jobs)\n    staged_jobs=[]\n    for idx,job in enumerate(jobs,1):\n        clone_name=""; staged_clone=None\n        if job.get("clone_wav"):\n            clone_src=Path(job["clone_wav"]); clone_name=clone_src.name; staged_clone=inp_dir/f"{idx:03d}_{sanitize_name(clone_src.name)}"; shutil.copy2(clone_src, staged_clone)\n        staged_jobs.append({"name":sanitize_name(job["name"]),"language":job.get("language") or default_language,"script":job["script"],"clone_wav":str(staged_clone) if staged_clone else None,"clone_wav_name":clone_name})\n    write_json(inp_dir/"jobs.json",staged_jobs); write_json(inp_dir/"profile.json",profile)\n    rows=[]; generated_paths=[]\n    for idx,job in enumerate(staged_jobs,1):\n        progress((idx-1)/max(1,len(staged_jobs)), desc=f"Generating {job[\'name\']} [{job[\'language\']}] ({idx}/{len(staged_jobs)})")\n        jp=dict(profile); jp["language"]=job["language"]\n        out_wav, section_logs=synthesize_one(job["name"],job["script"],job["clone_wav"],jp,run_dir,per_job_log_dir)\n        generated_paths.append(str(out_wav))\n        write_json(per_job_log_dir/f"{job[\'name\']}.json",{"sections":section_logs,"chars":len(strip_markers_for_model(job["script"])),"clone_wav_name":job["clone_wav_name"],"language":job["language"],"output_wav":str(out_wav)})\n        rows.append({"name":job["name"],"language":job["language"],"chars":len(strip_markers_for_model(job["script"])),"eta_s":estimate_seconds(job["script"]),"clone_wav":job["clone_wav_name"],"file":str(out_wav)})\n    upload_retry_attempts=int(os.environ.get("UPLOAD_RETRY_ATTEMPTS","3") or "3")\n    destroy_on_upload_failure=os.environ.get("DESTROY_ON_UPLOAD_FAILURE","true").lower()=="true"\n    upload_log={"uploaded":False,"upload_retry_attempts":upload_retry_attempts,"destroy_on_upload_failure":destroy_on_upload_failure}\n    if auto_upload:\n        remote=os.environ.get("RCLONE_REMOTE","").strip()\n        if remote:\n            ok_inputs=retry_rclone_copy(inp_dir,f"{remote}/{run_id}/inputs","inputs",upload_retry_attempts,upload_log)\n            ok_outputs=retry_rclone_copy(run_dir,f"{remote}/{run_id}/outputs","outputs",upload_retry_attempts,upload_log)\n            ok_logs=retry_rclone_copy(log_dir,f"{remote}/{run_id}/logs","logs",upload_retry_attempts,upload_log)\n            upload_log["uploaded"]=bool(ok_inputs and ok_outputs and ok_logs)\n            if not upload_log["uploaded"]:\n                salvage={}\n                salvage_ok=retry_rclone_copy(log_dir,f"{remote}/{run_id}/logs_failure_salvage","logs_failure_salvage",max(1,upload_retry_attempts),salvage)\n                upload_log["failure_log_salvage"]=salvage.get("logs_failure_salvage",{})\n                upload_log["failure_log_salvage"]["ok"]=bool(salvage_ok)\n        else:\n            upload_log["error"]="RCLONE_REMOTE is empty"\n    write_json(log_dir/"upload.json",upload_log)\n    req_path=APP_ROOT/"requirements.lock.txt"\n    metadata={"run_id":run_id,"repo_url":os.environ.get("FORK_REPO_URL",""),"repo_commit_sha":os.environ.get("FORK_COMMIT_SHA",""),"hf_model_repo_id":os.environ.get("HF_MODEL_REPO_ID",""),"hf_model_revision":os.environ.get("HF_MODEL_REVISION",""),"requirements_lock_sha256":sha256_file(req_path) if req_path.exists() else "","gpu_name":GPU_NAME,"device":DEVICE,"started_utc":started_utc,"ended_utc":now_utc(),"rclone_remote":os.environ.get("RCLONE_REMOTE",""),"count_items":len(staged_jobs),"profile":profile,"generated_files":generated_paths,"jobs":staged_jobs}\n    write_json(log_dir/"RUN_METADATA.json",metadata)\n    elapsed=round(time.time()-started,2)\n    if destroy_after_run:\n        if upload_log.get("uploaded"):\n            schedule_vast_action("destroy" if os.environ.get("DESTROY_INSTEAD_OF_STOP","true").lower()=="true" else "stop", "upload_success")\n        elif destroy_on_upload_failure:\n            schedule_vast_action("destroy","upload_failed_after_retries")\n    summary={"run_id":run_id,"gpu":GPU_NAME,"items":len(staged_jobs),"elapsed_s":elapsed,"uploaded":upload_log.get("uploaded",False),"generated":generated_paths,"languages":sorted(list({j["language"] for j in staged_jobs})),"destroy_on_upload_failure":destroy_on_upload_failure}\n    write_json(log_dir/"summary.json",summary)\n    md=f"### Run summary\\n- **Run ID**: `{run_id}`\\n- **GPU**: `{GPU_NAME}`\\n- **Items**: `{len(staged_jobs)}`\\n- **Languages**: `{\', \'.join(summary[\'languages\'])}`\\n- **Elapsed**: `{elapsed}s`\\n- **Uploaded**: `{upload_log.get(\'uploaded\', False)}`\\n- **Destroy on upload failure**: `{destroy_on_upload_failure}`\\n"\n    return rows, generated_paths, json.dumps(summary, ensure_ascii=False, indent=2), md\n\nCSS=".gradio-container {max-width: 1550px !important;} .header-card {border-radius: 18px; padding: 18px; background: linear-gradient(135deg, #111827, #1f2937); color: white; margin-bottom: 12px;} .small-muted {opacity: .78; font-size: .92rem;}"\nexample_profile=json.dumps(DEFAULT_PROFILE, ensure_ascii=False, indent=2)\nwith gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:\n    jobs_state=gr.State([])\n    gr.HTML(f\'<div class="header-card"><h1>Chatterbox — Destroy After Failed Upload With Log Salvage</h1><div class="small-muted">GPU: {GPU_NAME} • Device: {DEVICE}</div></div>\')\n    with gr.Row():\n        with gr.Column(scale=2):\n            job_name=gr.Textbox(label="Name", placeholder="episode_001")\n            job_language=gr.Dropdown(choices=SUPPORTED_LANGUAGES, value="ja", label="Job language")\n            job_script=gr.Textbox(label="Script", lines=16, placeholder="[HOOK] ... [CONTEXT] ... [DEV] ... [CLIMAX] ... [END] ...")\n            job_clone=gr.File(label="Clone WAV", file_count="single", type="filepath")\n            with gr.Row():\n                add_btn=gr.Button("Add", variant="primary"); remove_btn=gr.Button("Remove Last"); clear_btn=gr.Button("Clear All")\n            queue_status=gr.Markdown("0 job(s) in queue.")\n        with gr.Column(scale=1):\n            default_language=gr.Dropdown(choices=SUPPORTED_LANGUAGES, value="ja", label="Default language")\n            voice_name=gr.Textbox(value="ja_female_neutral", label="Voice name")\n            profile_json=gr.Code(value=example_profile, language="json", label="Profile JSON")\n            auto_upload=gr.Checkbox(value=True, label="Upload inputs/outputs/logs with rclone")\n            destroy_after_run=gr.Checkbox(value=True, label="Destroy after success or failed upload retries")\n            gen_btn=gr.Button("Generate All", variant="primary")\n    jobs_table=gr.Dataframe(headers=["#","name","language","chars","has_clone","eta_s"], datatype=["number","str","str","number","bool","number"], label="Queued jobs", interactive=False)\n    result_table=gr.Dataframe(headers=["name","language","chars","eta_s","clone_wav","file"], datatype=["str","str","number","number","str","str"], label="Generated items", interactive=False)\n    gallery_audio=gr.File(label="Output WAV files", file_count="multiple")\n    summary_json=gr.Code(label="Summary JSON", language="json")\n    summary_md=gr.Markdown()\n    add_btn.click(fn=add_job, inputs=[job_name,job_language,job_script,job_clone,jobs_state], outputs=[jobs_state,jobs_table,queue_status,job_name,job_clone], queue=False)\n    remove_btn.click(fn=remove_last_job, inputs=[jobs_state], outputs=[jobs_state,jobs_table,queue_status], queue=False)\n    clear_btn.click(fn=clear_jobs, inputs=[], outputs=[jobs_state,jobs_table,queue_status], queue=False)\n    gen_btn.click(fn=generate_all, inputs=[jobs_state,default_language,voice_name,profile_json,auto_upload,destroy_after_run], outputs=[result_table,gallery_audio,summary_json,summary_md], queue=True)\nif __name__=="__main__":\n    log_line("App started"); demo.queue(default_concurrency_limit=1); demo.launch(server_name="0.0.0.0", server_port=int(os.environ.get("SERVER_PORT","7860")), share=False)\n',
    "setup_remote.sh": '#!/usr/bin/env bash\nset -euo pipefail\nexport DEBIAN_FRONTEND=noninteractive\n\nAPP_ROOT="${APP_ROOT:-/workspace/chbx_ja_batch}"\nFORK_REPO_URL="${FORK_REPO_URL:?FORK_REPO_URL is required}"\nFORK_COMMIT_SHA="${FORK_COMMIT_SHA:?FORK_COMMIT_SHA is required}"\nHF_MODEL_REPO_ID="${HF_MODEL_REPO_ID:-}"\nHF_MODEL_REVISION="${HF_MODEL_REVISION:-}"\nHF_CACHE_DIR="${HF_CACHE_DIR:-/workspace/.cache/huggingface}"\nCLOUDFLARED_BIN="${CLOUDFLARED_BIN:-/usr/local/bin/cloudflared}"\n\nmkdir -p "$APP_ROOT/runtime_logs"\nexec > >(tee -a "$APP_ROOT/runtime_logs/setup_remote.log") 2>&1\n\necho "[setup] starting"\necho "[setup] APP_ROOT=$APP_ROOT"\necho "[setup] FORK_REPO_URL=$FORK_REPO_URL"\necho "[setup] FORK_COMMIT_SHA=$FORK_COMMIT_SHA"\necho "[setup] HF_MODEL_REPO_ID=$HF_MODEL_REPO_ID"\necho "[setup] HF_MODEL_REVISION=$HF_MODEL_REVISION"\n\napt-get update\napt-get install -y git git-lfs ffmpeg curl ca-certificates tmux build-essential rclone\n\npython3 -m pip install --upgrade pip setuptools wheel\npython3 -m pip install -r "$APP_ROOT/requirements.lock.txt"\n\nif [ ! -d "$APP_ROOT/src/.git" ]; then\n  git clone "$FORK_REPO_URL" "$APP_ROOT/src"\nfi\n\ncd "$APP_ROOT/src"\ngit fetch --all --tags\ngit checkout "$FORK_COMMIT_SHA"\ngit rev-parse HEAD\npython3 -m pip install -e .\n\npython3 -m pip freeze | tail -n 250 > "$APP_ROOT/runtime_logs/pip_freeze_after_setup.log" || true\nrclone version > "$APP_ROOT/runtime_logs/rclone_version.log" 2>&1 || true\nnvidia-smi > "$APP_ROOT/runtime_logs/nvidia_smi_setup.log" 2>&1 || true\n\nif [ ! -x "$CLOUDFLARED_BIN" ]; then\n  ARCH="$(dpkg --print-architecture)"\n  case "$ARCH" in\n    amd64) CF_PKG="cloudflared-linux-amd64" ;;\n    arm64) CF_PKG="cloudflared-linux-arm64" ;;\n    *) echo "Unsupported arch for cloudflared: $ARCH"; exit 1 ;;\n  esac\n  curl -L "https://github.com/cloudflare/cloudflared/releases/latest/download/${CF_PKG}" -o "$CLOUDFLARED_BIN"\n  chmod +x "$CLOUDFLARED_BIN"\nfi\n\necho "[setup] complete"\n',
    "launch_ui.sh": '#!/usr/bin/env bash\nset -euo pipefail\n\nAPP_ROOT="${APP_ROOT:-/workspace/chbx_ja_batch}"\nSERVER_PORT="${SERVER_PORT:-7860}"\nPYTHON_BIN="${PYTHON_BIN:-python3}"\nLOG_DIR="$APP_ROOT/runtime_logs"\nmkdir -p "$LOG_DIR"\n\n{\n  echo "[launch] $(date -u +%FT%TZ)"\n  echo "[launch] APP_ROOT=$APP_ROOT"\n  echo "[launch] SERVER_PORT=$SERVER_PORT"\n  echo "[launch] RCLONE_REMOTE=${RCLONE_REMOTE:-}"\n  echo "[launch] VAST_INSTANCE_ID=${VAST_INSTANCE_ID:-}"\n  echo "[launch] DESTROY_INSTEAD_OF_STOP=${DESTROY_INSTEAD_OF_STOP:-}"\n  echo "[launch] FORK_REPO_URL=${FORK_REPO_URL:-}"\n  echo "[launch] FORK_COMMIT_SHA=${FORK_COMMIT_SHA:-}"\n} >> "$LOG_DIR/launch_ui.log"\n\ncd "$APP_ROOT"\nif ! command -v tmux >/dev/null 2>&1; then\n  apt-get update && apt-get install -y tmux\nfi\n\ntmux kill-session -t chbx-ui >/dev/null 2>&1 || true\ntmux new-session -d -s chbx-ui "cd \'$APP_ROOT\' && APP_ROOT=\'$APP_ROOT\' SERVER_PORT=\'$SERVER_PORT\' HF_MODEL_REPO_ID=\'${HF_MODEL_REPO_ID:-}\' HF_MODEL_REVISION=\'${HF_MODEL_REVISION:-}\' HF_CACHE_DIR=\'${HF_CACHE_DIR:-/workspace/.cache/huggingface}\' RCLONE_REMOTE=\'${RCLONE_REMOTE:-}\' VAST_INSTANCE_ID=\'${VAST_INSTANCE_ID:-}\' DESTROY_INSTEAD_OF_STOP=\'${DESTROY_INSTEAD_OF_STOP:-true}\' FORK_REPO_URL=\'${FORK_REPO_URL:-}\' FORK_COMMIT_SHA=\'${FORK_COMMIT_SHA:-}\' $PYTHON_BIN app.py >> \'$LOG_DIR/app.log\' 2>&1"\n\necho "APP_LOG=$LOG_DIR/app.log"\necho "APP_PORT=$SERVER_PORT"\n',
    "start_tunnel.sh": '#!/usr/bin/env bash\nset -euo pipefail\nAPP_ROOT="${APP_ROOT:-/workspace/chbx_ja_batch}"\nSERVER_PORT="${SERVER_PORT:-7860}"\nCLOUDFLARED_BIN="${CLOUDFLARED_BIN:-/usr/local/bin/cloudflared}"\nLOG_DIR="$APP_ROOT/runtime_logs"\nmkdir -p "$LOG_DIR"\n\necho "[tunnel] $(date -u +%FT%TZ) starting tunnel for port $SERVER_PORT" >> "$LOG_DIR/cloudflared_control.log"\nrm -f "$LOG_DIR/cloudflared.log"\nnohup "$CLOUDFLARED_BIN" tunnel --url "http://127.0.0.1:${SERVER_PORT}" > "$LOG_DIR/cloudflared.log" 2>&1 &\nsleep 5\n\npython3 - <<\'PY\'\nimport pathlib, re, time\np = pathlib.Path("/workspace/chbx_ja_batch/runtime_logs/cloudflared.log")\nfor _ in range(30):\n    if p.exists():\n        txt = p.read_text(encoding="utf-8", errors="ignore")\n        m = re.search(r"https://[a-zA-Z0-9.-]+trycloudflare\\.com", txt)\n        if m:\n            print(m.group(0))\n            raise SystemExit(0)\n    time.sleep(2)\nraise SystemExit("No trycloudflare URL found yet.")\nPY\n',
}
remote_base = CONFIG["remote_base_dir"].rstrip("/")
for name, content in bundle_files.items():
    remote.write_text(f"{remote_base}/{name}", content)
remote.run(f"chmod +x {remote_base}/*.sh")
print("Bundle copied to remote:", remote_base)

In [ ]:
if CONFIG["rclone_config_b64"].strip():
    raw = base64.b64decode(CONFIG["rclone_config_b64"])
    remote.write_bytes("/root/.config/rclone/rclone.conf", raw)
    remote.run("chmod 600 /root/.config/rclone/rclone.conf")
    print("rclone.conf written.")
else:
    print("No rclone_config_b64 provided. Assuming rclone is already configured on Vast.")

In [ ]:
env = {
    "APP_ROOT": CONFIG["remote_base_dir"],
    "FORK_REPO_URL": CONFIG["fork_repo_url"],
    "FORK_COMMIT_SHA": CONFIG["fork_commit_sha"],
    "HF_MODEL_REPO_ID": CONFIG["hf_model_repo_id"],
    "HF_MODEL_REVISION": CONFIG["hf_model_revision"],
    "HF_CACHE_DIR": CONFIG["hf_cache_dir"],
    "CLOUDFLARED_BIN": CONFIG["cloudflared_bin"],
}
res = remote.run(f"cd {shlex.quote(CONFIG['remote_base_dir'])} && bash setup_remote.sh", env=env, get_pty=True, timeout=3600)
print(res["stdout"][-12000:])
if res["stderr"].strip(): print(res["stderr"][-8000:])

In [ ]:
print(remote.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False)["stdout"])
print(remote.run("rclone lsd gdrive:", check=False)["stdout"])
print(remote.run(f"tail -n 100 {shlex.quote(CONFIG['remote_base_dir'])}/runtime_logs/setup_remote.log", check=False)["stdout"])

In [ ]:
launch_env = {
    "APP_ROOT": CONFIG["remote_base_dir"],
    "SERVER_PORT": str(CONFIG["server_port"]),
    "HF_MODEL_REPO_ID": CONFIG["hf_model_repo_id"],
    "HF_MODEL_REVISION": CONFIG["hf_model_revision"],
    "HF_CACHE_DIR": CONFIG["hf_cache_dir"],
    "RCLONE_REMOTE": CONFIG["rclone_remote"],
    "VAST_INSTANCE_ID": str(CONFIG["vast_instance_id"] or ""),
    "DESTROY_INSTEAD_OF_STOP": str(CONFIG["destroy_instead_of_stop"]).lower(),
    "DESTROY_ON_UPLOAD_FAILURE": str(CONFIG["destroy_on_upload_failure"]).lower(),
    "UPLOAD_RETRY_ATTEMPTS": str(CONFIG["upload_retry_attempts"]),
    "FORK_REPO_URL": CONFIG["fork_repo_url"],
    "FORK_COMMIT_SHA": CONFIG["fork_commit_sha"],
}
res = remote.run(f"cd {shlex.quote(CONFIG['remote_base_dir'])} && bash launch_ui.sh", env=launch_env, get_pty=True)
print(res["stdout"])

In [ ]:
port = int(CONFIG["server_port"])
health_cmd = f"""python3 - <<'PY'
import time, requests
url = "http://127.0.0.1:{port}"
for _ in range(60):
    try:
        r = requests.get(url, timeout=3)
        print("HTTP", r.status_code)
        raise SystemExit(0)
    except Exception:
        time.sleep(2)
raise SystemExit("App did not become ready in time.")
PY"""
print(remote.run(health_cmd, check=False)["stdout"])

In [ ]:
out = remote.run(
    f"cd {shlex.quote(CONFIG['remote_base_dir'])} && bash start_tunnel.sh",
    env={"APP_ROOT": CONFIG["remote_base_dir"], "SERVER_PORT": str(CONFIG["server_port"]), "CLOUDFLARED_BIN": CONFIG["cloudflared_bin"]},
    check=False,
    get_pty=True,
    timeout=180,
)
print(out["stdout"])
match = re.search(r"https://[a-zA-Z0-9.-]+trycloudflare\.com", out["stdout"] + "\n" + out["stderr"])
print("PUBLIC_URL =", match.group(0) if match else None)

In [ ]:
print(remote.run(f"tail -n 120 {shlex.quote(CONFIG['remote_base_dir'])}/runtime_logs/app.log", check=False)["stdout"])
print(remote.run(f"tail -n 120 {shlex.quote(CONFIG['remote_base_dir'])}/runtime_logs/debug_app.log", check=False)["stdout"])
print(remote.run(f"tail -n 120 {shlex.quote(CONFIG['remote_base_dir'])}/runtime_logs/cloudflared.log", check=False)["stdout"])